In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy pandas
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

import TutorialFeedback from '@site/src/components/TutorialFeedback';



*Geschätzter Aufwand: unter einer Minute auf einem Heron-r3-Prozessor (HINWEIS: Dies ist nur eine Schätzung. Deine tatsächliche Laufzeit kann abweichen.)*

## Lernziele

Nach Abschluss dieses Tutorials kannst du folgende Informationen nachvollziehen:

  * Kernel-Methoden und ihre Anwendungen
  * Quantenkerne und wie sie erweiterte Feature-Räume bereitstellen können
  * Aufbau von Quantenkern-Circuits
  * Training eines Quantenkerns mithilfe eines [Qiskit-Patterns](/guides/intro-to-patterns): Abbilden, Optimieren, Ausführen und Nachverarbeiten

## Voraussetzungen

Es wird empfohlen, dich mit Quantenkernen vertraut zu machen, warum sie wichtig sind und wie sie in der Praxis eingesetzt werden.

  * [Covariant quantum kernels for data with group structure](https://arxiv.org/abs/2105.03406) (Fachartikel)
  * [Introduction to Quantum Kernels and Support Vector Machines](https://www.youtube.com/watch?v=GVhCOTzAkCM) (Video)
  * [Quantum Kernels in Practice](https://www.youtube.com/watch?v=LmQcSxgINis) (Video)

Außerdem ist ein grundlegendes Verständnis der Gruppentheorie hilfreich.

## Hintergrund

Kernel-Methoden sind in maschinellen Lernanwendungen weit verbreitet.
In diesem Zusammenhang bezeichnet „Kernel" die Kernmatrix oder einzelne Einträge darin.
Im Allgemeinen ist ein Kernel ein Ähnlichkeitsmaß zwischen Daten, die in einem hochdimensionalen _Feature-Raum_ kodiert sind, und kann beispielsweise bei Klassifikationsaufgaben mit Support-Vektor-Maschinen eingesetzt werden.

Quantenkern-Methoden sind solche, die Quantencomputer zur Schätzung eines Kernels verwenden.
Es ist bekannt, dass Quantencomputer Daten in quantenverstärkten Feature-Räumen kodieren können und so klassische Pendants effektiv ersetzen.
Für $\vec{x} \in \mathbb{R}$ und $\Psi(\vec{x}) \in \mathbb{R}^{d'}$, typischerweise mit $d' >d$, ist $\Psi(\vec{x})$ eine Feature-Map, $\vec{x} \mapsto \Psi(\vec{x})$.
Das Ziel von $\Psi(\vec{x})$ ist es, Datenkategorien durch eine Hyperebene zu trennen.
Die Kernfunktion $K(\vec{x}, \vec{y}) = \langle{\Psi(\vec{x}) | \Psi(\vec{y}) \rangle{}}$ nimmt die Vektoren im Feature-abgebildeten Raum als Argumente und gibt ihr inneres Produkt zurück: $K: \mathbb{R}^d \rightarrow$ $\mathbb{R}^d$.
Klassisch sind Feature-Maps von Interesse, bei denen die Kernfunktion leicht ausgewertet werden kann;
das heißt, wenn das innere Produkt im Feature-abgebildeten Raum in Termen der ursprünglichen Datenvektoren geschrieben werden kann und $\Psi(\vec{x})$ und $\Psi(\vec{y})$ nicht explizit konstruiert werden müssen.
Bei Quantenkernen wird die Feature-Abbildung durch einen Quantencircuit durchgeführt, und der Kernel wird anhand der Messprobabilitäten geschätzt, die aus dem Circuit gesampelt werden.

Dieses Tutorial zeigt, wie du ein Qiskit-Pattern erstellst, um Einträge einer Quantenkernmatrix für binäre Klassifikation zu berechnen.

## Anforderungen

Stelle vor dem Start dieses Tutorials sicher, dass folgende Pakete installiert sind:
- Qiskit SDK v2.3.1 oder höher, mit Unterstützung für [Visualisierung](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.44.0 oder höher (`pip install qiskit-ibm-runtime`)

## Einrichtung

In [ ]:
# General Imports and helper functions
import urllib.request

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


from qiskit.circuit import Parameter, ParameterVector, QuantumCircuit
from qiskit.circuit.library import unitary_overlap
from qiskit.primitives import StatevectorSampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService, Sampler

# Download the dataset (portable across platforms)
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/qiskit-community/prototype-quantum-kernel-training/main/data/dataset_graph7.csv",
    "dataset_graph7.csv",
)


def visualize_counts(res_counts, num_qubits, num_shots):
    """Visualize the outputs from the Qiskit Sampler primitive."""
    zero_prob = res_counts.get(0, 0.0)
    top_10 = dict(
        sorted(res_counts.items(), key=lambda item: item[1], reverse=True)[
            :10
        ]
    )
    top_10.update({0: zero_prob})
    by_key = dict(sorted(top_10.items(), key=lambda item: item[0]))
    x_vals, y_vals = list(zip(*by_key.items()))
    x_vals = [bin(x_val)[2:].zfill(num_qubits) for x_val in x_vals]
    y_vals_prob = []
    for t in range(len(y_vals)):
        y_vals_prob.append(y_vals[t] / num_shots)
    y_vals = y_vals_prob
    plt.bar(x_vals, y_vals)
    plt.xticks(rotation=75)
    plt.title("Results of sampling")
    plt.xlabel("Measured bitstring")
    plt.ylabel("Probability")
    plt.show()


def get_training_data():
    """Read the training data."""
    df = pd.read_csv("dataset_graph7.csv", sep=",", header=None)
    training_data = df.values[:20, :]
    ind = np.argsort(training_data[:, -1])
    X_train = training_data[ind][:, :-1]

    return X_train

## Kleines Simulatorbeispiel

In diesem Abschnitt gehen wir die vier Schritte des Qiskit-Patterns für eine Sieben-Qubit-Instanz des Problems „Labeling Cosets with Error" durch und werten einen einzelnen Kernmatrixeintrag mit dem `StatevectorSampler`-Primitiv aus Qiskit aus. Ein Zustandsvektorsimulator ist exakt (abgesehen von Shot-Rauschen) und zeigt uns die Methode von Anfang bis Ende, ohne QPU-Zeit zu verbrauchen. Wir wiederholen dieselbe Instanz dann auf echter Hardware im Abschnitt zum Hardware-Beispiel.

### Schritt 1: Klassische Eingaben auf ein Quantenproblem abbilden

*   Eingabe: Trainingsdatensatz.
*   Ausgabe: Abstrakter Circuit zur Berechnung eines Kernmatrixeintrags.

Das binäre Klassifikationsproblem, das wir hier lösen möchten, wird als „[_Labeling Cosets with Error_](https://arxiv.org/abs/2105.03406)" bezeichnet. Der Eingabe-Trainingsdatensatz enthält eine Gruppenstruktur, bestehend aus zwei Nebenklassen, die durch eine Gruppe und eine Untergruppe gebildet werden.
Die Gruppe ist $G = SU(2)^{\otimes n}$ für Qubits, d. h. die spezielle unitäre Gruppe der
$2 \times 2$-Matrizen, die in der Natur weit verbreitet ist; z. B. im Standardmodell der Teilchenphysik.
Wir wählen die (Graph-Stabilisator-)Untergruppe $S_\text{graph} < G$ mit $S_\text{graph} = \langle { X_i \otimes _{k:(k,i) \in \mathcal{E}} Z_k} _{i \in \mathcal{V}} } \rangle$ für einen Graphen mit Kanten $\mathcal{E}$ und Knoten $\mathcal{V}$.
Beachte, dass die Stabilisatoren einen Stabilisatorzustand fixieren, sodass $D_s | \psi \rangle = | \psi \rangle,~ \forall s \in S_\text{graph}$.
Schließlich definieren wir zwei linke Nebenklassen $C_\pm = c_\pm S_\text{graph}$, indem wir zwei $c_\pm \in G$ zufällig ziehen.

Weitere Details zum Datensatz und seiner Generierung findest du in [diesem Notebook](https://github.com/qiskit-community/prototype-quantum-kernel-training/blob/main/docs/background/qkernels_and_data_w_group_structure.ipynb) aus dem [Quantum Kernel Training Toolkit](https://github.com/qiskit-community/prototype-quantum-kernel-training/tree/main).

Wir erstellen den Quantum Circuit, der zur Auswertung eines Eintrags in der Kernmatrix verwendet wird.
Die Eingabedaten werden genutzt, um die Rotationswinkel der parametrisierten Gates des Circuits zu bestimmen.
Der Einfachheit halber verwenden wir die Datenpunkte `x1=14` und `x2=19`.

***Hinweis: Der in diesem Tutorial verwendete Datensatz kann [hier](https://github.com/qiskit-community/prototype-quantum-kernel-training/blob/main/data/dataset_graph7.csv) heruntergeladen werden.***

In [ ]:
# Prepare training data
X_train = get_training_data()

# Empty kernel matrix
num_samples = np.shape(X_train)[0]
kernel_matrix = np.full((num_samples, num_samples), np.nan)

# Prepare feature map for computing overlap
num_features = np.shape(X_train)[1]
num_qubits = int(num_features / 2)
entangler_map = [[0, 2], [3, 4], [2, 5], [1, 4], [2, 3], [4, 6]]
fm = QuantumCircuit(num_qubits)
training_param = Parameter("θ")
feature_params = ParameterVector("x", num_qubits * 2)
fm.ry(training_param, fm.qubits)
for cz in entangler_map:
    fm.cz(cz[0], cz[1])
for i in range(num_qubits):
    fm.rz(-2 * feature_params[2 * i + 1], i)
    fm.rx(-2 * feature_params[2 * i], i)

# Assign tunable parameter to known optimal value and set the data params for
# first two samples
x1 = 14
x2 = 19
unitary1 = fm.assign_parameters(list(X_train[x1]) + [np.pi / 2])
unitary2 = fm.assign_parameters(list(X_train[x2]) + [np.pi / 2])

# Create the overlap circuit
overlap_circ = unitary_overlap(unitary1, unitary2)
overlap_circ.measure_all()
overlap_circ.draw("mpl", scale=0.6, style="iqp")

<Image src="../docs/images/tutorials/quantum-kernel-training/extracted-outputs/70d6faff-9a56-44bb-b26f-f573a8c90889-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/quantum-kernel-training/extracted-outputs/70d6faff-9a56-44bb-b26f-f573a8c90889-0.avif)

### Schritt 2: Problem für die Ausführung auf Quantenhardware optimieren
*   Eingabe: Abstrakter Circuit, nicht für ein bestimmtes Backend optimiert.
*   Ausgabe: Ziel-Circuit, optimiert für den ausgewählten QPU.

Für den in diesem Abschnitt verwendeten Zustandsvektorsimulator ist keine Backend-spezifische Optimierung erforderlich: Der abstrakte Circuit kann direkt gesampelt werden. Wir führen diesen Schritt im Hardware-Beispiel unten aus, wo der Circuit mit `generate_preset_pass_manager` mit `optimization_level=3` gegen einen echten QPU transpiliert wird.
### Schritt 3: Ausführung mit Qiskit-Primitiven
*   Eingabe: Abstrakter Circuit.
*   Ausgabe: Quasi-Wahrscheinlichkeitsverteilung.

Verwende das `StatevectorSampler`-Primitiv aus Qiskit, um eine Quasi-Wahrscheinlichkeitsverteilung der Zustände zu rekonstruieren, die beim Sampling des Circuits entstehen. Für die Erstellung einer Kernmatrix interessiert uns besonders die Wahrscheinlichkeit, den Zustand |0> zu messen.

In [3]:
sampler = StatevectorSampler()

# Execute and get counts
num_shots = 10_000
results = sampler.run([overlap_circ], shots=num_shots).result()
counts = results[0].data.meas.get_int_counts()

# Plot counts
visualize_counts(counts, num_qubits, num_shots)

<Image src="../docs/images/tutorials/quantum-kernel-training/extracted-outputs/step3-small-code-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/quantum-kernel-training/extracted-outputs/step3-small-code-0.avif)

### Schritt 4: Nachverarbeitung und Rückgabe des Ergebnisses im gewünschten klassischen Format
*   Eingabe: Wahrscheinlichkeitsverteilung.
*   Ausgabe: Ein einzelnes Kernmatrixelement.

Berechne die Wahrscheinlichkeit, $|0 \rangle$ auf dem Overlap-Circuit zu messen, und befülle die Kernmatrix an der Position, die den durch diesen Circuit repräsentierten Datenpunkten entspricht (Zeile 15, Spalte 20).

In [4]:
kernel_matrix[x1, x2] = counts.get(0, 0.0) / num_shots
print(f"Fidelity (simulator): {kernel_matrix[x1, x2]}")

Fidelity (simulator): 0.8261


## Hardware example

A quantum kernel matrix has $\mathcal{O}(N^2)$ entries for $N$ training samples, and each entry requires running an overlap circuit whose two-qubit-gate depth grows with the size of the feature map. As a result, scaling this tutorial to a larger problem has two compounding costs: the QPU time per kernel matrix grows quadratically with $N$, and the depth of `unitary_overlap` (which composes the feature map with its adjoint) erodes fidelity at the system size and connectivity of current hardware. To keep the demo short and to make a clean comparison, we therefore run the **same** seven-qubit instance from the small-scale example on a real QPU and compare the fidelity of a single kernel matrix entry against the simulator value computed above.

In [ ]:
# ------------------------------ Step 1 ------------------------------
# Prepare training data
X_train = get_training_data()

# Empty kernel matrix
num_samples = np.shape(X_train)[0]
kernel_matrix = np.full((num_samples, num_samples), np.nan)

# Prepare feature map for computing overlap
num_features = np.shape(X_train)[1]
num_qubits = int(num_features / 2)
entangler_map = [[0, 2], [3, 4], [2, 5], [1, 4], [2, 3], [4, 6]]
fm = QuantumCircuit(num_qubits)
training_param = Parameter("θ")
feature_params = ParameterVector("x", num_qubits * 2)
fm.ry(training_param, fm.qubits)
for cz in entangler_map:
    fm.cz(cz[0], cz[1])
for i in range(num_qubits):
    fm.rz(-2 * feature_params[2 * i + 1], i)
    fm.rx(-2 * feature_params[2 * i], i)

# Assign tunable parameter to known optimal value and
# set the data params for first two samples
x1 = 14
x2 = 19
unitary1 = fm.assign_parameters(list(X_train[x1]) + [np.pi / 2])
unitary2 = fm.assign_parameters(list(X_train[x2]) + [np.pi / 2])

# Create the overlap circuit
overlap_circ = unitary_overlap(unitary1, unitary2)
overlap_circ.measure_all()

# ------------------------------ Step 2 ------------------------------
service = QiskitRuntimeService()
# backend = service.least_busy(
#    operational=True, simulator=False, min_num_qubits=overlap_circ.num_qubits
# )
backend = service.backend("ibm_pittsburgh")
print(f"Using backend: {backend.name}")
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)
overlap_ibm = pm.run(overlap_circ)

# ------------------------------ Step 3 ------------------------------
sampler = Sampler(mode=backend)
sampler.options.environment.job_tags = ["TUT_QKT"]

num_shots = 10_000
results = sampler.run([overlap_ibm], shots=num_shots).result()
counts = results[0].data.meas.get_int_counts()
visualize_counts(counts, num_qubits, num_shots)

# ------------------------------ Step 4 ------------------------------
kernel_matrix[x1, x2] = counts.get(0, 0.0) / num_shots
print(f"Fidelity (hardware): {kernel_matrix[x1, x2]}")

Using backend: ibm_pittsburgh


<Image src="../docs/images/tutorials/quantum-kernel-training/extracted-outputs/d2f4f6cf-067e-4d53-aa04-7ca9c803d3e1-1.avif" alt="Output of the previous code cell" />

Fidelity (hardware): 0.7517


## Hardware-Beispiel
Eine Quantenkernmatrix hat $\mathcal{O}(N^2)$ Einträge für $N$ Trainingssamples, und jeder Eintrag erfordert die Ausführung eines Overlap-Circuits, dessen Tiefe der Zwei-Qubit-Gates mit der Größe der Feature-Map wächst. Das Skalieren dieses Tutorials auf ein größeres Problem hat daher zwei miteinander verknüpfte Kosten: Die QPU-Zeit pro Kernmatrix wächst quadratisch mit $N$, und die Tiefe von `unitary_overlap` (das die Feature-Map mit ihrer Adjungierten verknüpft) verringert die Fidelität bei der Systemgröße und Konnektivität aktueller Hardware. Um die Demo kurz zu halten und einen sauberen Vergleich zu ermöglichen, führen wir daher dieselbe Sieben-Qubit-Instanz aus dem kleinen Beispiel auf einem echten QPU aus und vergleichen die Fidelität eines einzelnen Kernmatrixeintrags mit dem oben berechneten Simulatorwert.